In [1]:
from pathlib import Path
import pandas as pd
import evaluation

In [2]:
meas_dir = Path("measurements")

measurement_files = {}
for file in meas_dir.glob("*.csv"):
    name, lower, upper, step, cycles, delay = file.name.split("_")
    stroke = float(upper) - float(lower)
    measurement_files[name] = sorted(measurement_files.get(name, []) + [(file, stroke)], key=lambda x: x[1])

characterizations: list[evaluation.Characterization] = []
for name, files in measurement_files.items():
    (short, _), (long, _) = files
    characterizations.append(evaluation.Characterization(short, long))

In [3]:
results = [char.summary(short_evaluation_range=5e-3, long_evaluation_range=50e-3) for char in characterizations]

In [ ]:
name_map = {
    "min_increment": "95% min. inc (µdeg)",
    "unidirectional_repeatability": "95% uni. rep. (µdeg)",
    "backlash": "backlash (µdeg)",
    "backlash_variation": "95% BL var. (µdeg)",
    "nonlinearity": "95% nonlin. (µdeg)",
}
order = ["cheap-geared", "cheap-direct", "pancake-geared", "pancake-direct", "newport-dc", "newport-stepper"]

def to_micro_deg(value: evaluation.UncertainValue) -> str:
    return f"{int(value.value * 1e6)} ± {int(value.max_deviation * 1e6)}"

results_df = pd.DataFrame(results).set_index("name").loc[order].rename(columns=name_map).map(to_micro_deg)
results_df

,95% min. inc (µdeg),95% uni. rep. (µdeg),95% non-lin. (µdeg),backlash (µdeg),95% BL var. (µdeg)
name,,,,,
cheap-geared,521 ± 27,490 ± 19,1034 ± 23,2763 ± 116,507 ± 19
cheap-direct,490 ± 39,2014 ± 131,923 ± 44,4420 ± 590,2007 ± 132
pancake-geared,445 ± 18,1144 ± 25,803 ± 40,1620 ± 768,1154 ± 20
pancake-direct,503 ± 33,614 ± 21,613 ± 20,1543 ± 170,622 ± 20
newport-dc,285 ± 15,534 ± 18,587 ± 18,10692 ± 105,539 ± 17
newport-stepper,298 ± 16,441 ± 17,533 ± 16,10049 ± 63,459 ± 14


In [9]:
print(results_df.to_markdown())

| name            | 95% min. inc (µdeg)   | 95% uni. rep. (µdeg)   | 95% non-lin. (µdeg)   | backlash (µdeg)   | 95% BL var. (µdeg)   |
|:----------------|:----------------------|:-----------------------|:----------------------|:------------------|:---------------------|
| cheap-geared    | 521 ± 27              | 490 ± 19               | 1034 ± 23             | 2763 ± 116        | 507 ± 19             |
| cheap-direct    | 490 ± 39              | 2014 ± 131             | 923 ± 44              | 4420 ± 590        | 2007 ± 132           |
| pancake-geared  | 445 ± 18              | 1144 ± 25              | 803 ± 40              | 1620 ± 768        | 1154 ± 20            |
| pancake-direct  | 503 ± 33              | 614 ± 21               | 613 ± 20              | 1543 ± 170        | 622 ± 20             |
| newport-dc      | 285 ± 15              | 534 ± 18               | 587 ± 18              | 10692 ± 105       | 539 ± 17             |
| newport-stepper | 298 ± 16              | 441 